In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import joblib
import os

# 1. Load the raw data
# Assuming your raw data is in data/raw/music_dataset.csv
df_raw = pd.read_csv('../data/raw/music_dataset.csv')

print(f"Original shape: {df_raw.shape}")
df_raw.head()

Original shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [8]:
# 2. Handle Missing Values
# For numerical audio features, fill with the median (more robust to outliers than mean)
numeric_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness', 
                'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

for col in numeric_cols:
    df_raw[col].fillna(df_raw[col].median(), inplace=True)

# For categorical, fill with 'Unknown' or mode
df_raw['track_genre'].fillna('unknown', inplace=True)
df_raw['artists'].fillna('unknown', inplace=True)

# 3. Fix Data Types
# 'explicit' is currently string 'FALSE'/'TRUE', convert to boolean/int (0/1)
df_raw['explicit'] = df_raw['explicit'].map({'TRUE': 1, 'FALSE': 0}).fillna(0).astype(int)

# Convert scientific notation columns to standard float (pandas usually handles this, but let's be safe)
df_raw['instrumentalness'] = df_raw['instrumentalness'].astype(float)

# Remove exact duplicate rows and duplicate track_ids
df_raw.drop_duplicates(inplace=True)
df_raw = df_raw.drop_duplicates(subset=['track_id'], keep='first')

print(f"Shape after removing duplicates: {df_raw.shape}")

print("Cleaning complete. Missing values check:")
print(df_raw.isnull().sum().sum()) # Should be 0

C:\Users\Kshitij\AppData\Local\Temp\ipykernel_2196\3164475115.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/docs/dev/user_guide/migration.html#copy-on-write-cow
  df_raw[col].fillna(df_raw[col].median(), inplace=True)
C:\Users\Kshitij\AppData\Local\Temp\ipykernel_2196\3164475115.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chaine

Shape after removing duplicates: (89741, 22)
Cleaning complete. Missing values check:
3


In [9]:
# We will use Label Encoding for high-cardinality or ordinal-like features, 
# and One-Hot Encoding for low-cardinality nominal features.

# 1. Label Encoding for 'track_genre' (since there might be many genres)
le_genre = LabelEncoder()
df_raw['track_genre_encoded'] = le_genre.fit_transform(df_raw['track_genre'])

# 2. One-Hot Encoding for 'key', 'mode', 'time_signature', 'explicit'
# These have very few unique values (e.g., mode is just 0 or 1, key is 0-11)
df_processed = pd.get_dummies(df_raw, columns=['key', 'mode', 'time_signature', 'explicit'], drop_first=False)

# Drop original text columns that we don't need for the model, but KEEP track_id for mapping later!
cols_to_drop = ['artists', 'album_name', 'track_name', 'track_genre']
df_model = df_processed.drop(columns=cols_to_drop)

print(f"Shape after encoding: {df_model.shape}")

Shape after encoding: (89741, 34)


In [10]:
# Separate the ID from the features
track_ids = df_model['track_id'].values
df_features = df_model.drop(columns=['track_id'])

# Initialize MinMaxScaler (scales everything between 0 and 1)
scaler = MinMaxScaler()

# Fit and transform the features
features_scaled = scaler.fit_transform(df_features)

# Convert back to a DataFrame for easier handling
df_final_features = pd.DataFrame(features_scaled, columns=df_features.columns)
df_final_features['track_id'] = track_ids

print("Scaling complete.")
df_final_features.head()

Scaling complete.


,Unnamed: 0,popularity,duration_ms,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,...,key_11,mode_0,mode_1,time_signature_0,time_signature_1,time_signature_3,time_signature_4,time_signature_5,explicit_0,track_id
0,0.000000,0.73,0.044043,0.686294,0.4610,0.791392,0.148187,0.032329,0.000001,0.3580,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,5SuOikwiRyPMVoIQDJUgSV
1,0.000009,0.55,0.028566,0.426396,0.1660,0.597377,0.079067,0.927711,0.000006,0.1010,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,4qPNDBW1i3p13qLCt0Ki3A
2,0.000018,0.57,0.040255,0.444670,0.3590,0.736123,0.057720,0.210843,0.000000,0.1170,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1iJBSr7s7jYXzM8EGcbK5b
3,0.000026,0.71,0.038557,0.270051,0.0596,0.573701,0.037617,0.908635,0.000071,0.1320,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,6lfxq3CG4xtTiEg7opyCyx
4,0.000035,0.82,0.037969,0.627411,0.4430,0.737103,0.054508,0.470884,0.000000,0.0829,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,5vjLSffimiIP26QG5WcN2K


In [11]:
# 1. Create a mapping dictionary: track_id -> {track_name, artists, album_name}
# We use the original raw dataframe for this
track_mapping = df_raw.set_index('track_id')[['track_name', 'artists', 'album_name', 'track_genre']].to_dict('index')

# 2. Create a reverse mapping: Index -> track_id (Useful for Markov Chains/Matrix operations)
# We reset the index so the row number becomes the index
df_final_features.reset_index(drop=True, inplace=True)
index_to_trackid = dict(enumerate(df_final_features['track_id']))
trackid_to_index = {v: k for k, v in index_to_trackid.items()}

# 3. Save everything to the 'data/processed/' and 'saved_models/' folders
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../saved_models/preprocessors', exist_ok=True)

# Save processed features (as CSV or Parquet - Parquet is much faster/lighter!)
df_final_features.to_parquet('../data/processed/processed_features.parquet', index=False)

# Save mappings
joblib.dump(track_mapping, '../saved_models/preprocessors/track_mapping.pkl')
joblib.dump(index_to_trackid, '../saved_models/preprocessors/index_to_trackid.pkl')
joblib.dump(trackid_to_index, '../saved_models/preprocessors/trackid_to_index.pkl')

# Save the fitted scaler (so the Streamlit app can scale new inputs exactly the same way)
joblib.dump(scaler, '../saved_models/preprocessors/minmax_scaler.pkl')
joblib.dump(le_genre, '../saved_models/preprocessors/label_encoder_genre.pkl')

print("Phase 2 Complete! Data saved successfully.")

C:\Users\Kshitij\AppData\Local\Temp\ipykernel_2196\3075623038.py:7: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_final_features.reset_index(drop=True, inplace=True)


Phase 2 Complete! Data saved successfully.
